# 🧠 Fine-tune Vio's real model — free on a Colab T4

This teaches a model that **already understands language** (Qwen2.5-7B) *your* networking &
security knowledge, using a **QLoRA** (4-bit) fine-tune. It runs on a **free T4 GPU** — about
**20× faster than your laptop, $0** — and produces a model that reasons *and* knows your domain.

The result exports to **GGUF** and runs locally in **Ollama**, offline, like any other model.

**First:** top menu → **Runtime → Change runtime type → T4 GPU → Save.** Then **Runtime → Run all.**

⏱ ~1–2 hours total on a free T4 (install + train + GGUF export). Keep the tab open.


### 1) Confirm the free GPU is attached


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → T4 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0), '·', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')


### 2) Install Unsloth
Unsloth makes a 7B QLoRA fit and train fast on a 16 GB T4. (~3–5 min.)


In [ ]:
%%capture
# Unsloth auto-detects Colab. If this ever fails, use the pinned fallback in the next comment.
!pip install unsloth
# fallback: !pip install --no-deps 'git+https://github.com/unslothai/unsloth.git' && pip install unsloth_zoo


### 3) Upload your training data — `vio_sft.jsonl`
**First, on your PC**, generate it with your own local model (this is the key step —
it turns your facts + ingested documents into thousands of rich, clean pairs):
```
cd AI-Foundation-Model-Design\mind
python build_dataset.py
```
That writes **`vio_sft.jsonl`**. Then run this cell, click **Choose Files**, and pick it.

*(Why not the tiny one-liner facts from before? Training on ~400 terse sentences overfit and*
*broke the model. This larger, richer, multi-sentence data is the fix.)*


In [ ]:
import json
from google.colab import files
up = files.upload()                         # pick vio_sft.jsonl
fn = [k for k in up if k.endswith('.jsonl')][0]
rows = [json.loads(l) for l in open(fn, encoding='utf-8') if l.strip()]
pairs = [{'q':r['question'], 'a':r['answer']} for r in rows
         if r.get('question') and r.get('answer')]
assert len(pairs) >= 200, ('Only %d pairs — generate more first: run build_dataset.py on all your datasets + ingested docs. Too little data is what broke the last run.' % len(pairs))
print(f'{len(pairs)} training pairs loaded')
print('example:', pairs[0])


### 4) Load Qwen2.5-7B (4-bit) and attach a LoRA
The base model is frozen; only the small LoRA adapters train — that's why 7B fits on a free T4.
*(Prefer Llama? swap the model name for* `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`*.)*


In [ ]:
from unsloth import FastLanguageModel
import torch
MAXLEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length = MAXLEN, dtype = None, load_in_4bit = True)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0, bias = 'none',
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth', random_state = 42)


### 5) Format with the chat template and train
**1 epoch, low learning rate — deliberately gentle.** The last fine-tune broke by running
3 epochs on tiny data and overfitting. With this larger, richer dataset, one careful pass
teaches the domain without wrecking the model's general reasoning. Watch the loss fall.


In [ ]:
from datasets import Dataset
def fmt(ex):
    msgs = [{'role':'user','content':ex['q']}, {'role':'assistant','content':ex['a']}]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}
ds = Dataset.from_list(pairs).map(fmt)

from trl import SFTTrainer
from transformers import TrainingArguments
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    dataset_text_field = 'text', max_seq_length = MAXLEN, packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 10, num_train_epochs = 1, learning_rate = 1e-4,   # gentle: avoids overfit
        fp16 = not torch.cuda.is_bf16_supported(), bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10, optim = 'adamw_8bit', weight_decay = 0.01,
        lr_scheduler_type = 'linear', seed = 42, output_dir = 'outputs', report_to = 'none'))
trainer.train()


### 6) Quick sanity check
Ask it something from your domain — it should answer in your material's voice.


In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{'role':'user','content':'What is a VDOM in FortiGate and when would I use one?'}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=220, temperature=0.6)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))


### 7) Export to GGUF and download
Merges the LoRA into the base and quantizes to a single file Ollama can run. (~15–25 min.)


In [ ]:
model.save_pretrained_gguf('vio_model', tokenizer, quantization_method='q4_k_m')
import glob, os
from google.colab import files
# Unsloth may save under vio_model/ or vio_model_gguf/ — search everywhere.
cands = glob.glob('**/*Q4_K_M.gguf', recursive=True) or glob.glob('**/*.gguf', recursive=True)
assert cands, 'No .gguf produced — check the conversion log above.'
gguf = cands[0]; fname = os.path.basename(gguf)
print('GGUF:', gguf, '·', round(os.path.getsize(gguf)/1e9,2), 'GB')
# Write a CORRECT Qwen2.5 (ChatML) Modelfile. Without this template Ollama can't
# format prompts and role labels like 'system' leak into the output as garbage.
mf = ['FROM ./' + fname, '',
      'TEMPLATE """{{ if .System }}<|im_start|>system',
      '{{ .System }}<|im_end|>',
      '{{ end }}{{ if .Prompt }}<|im_start|>user',
      '{{ .Prompt }}<|im_end|>',
      '{{ end }}<|im_start|>assistant',
      '{{ .Response }}<|im_end|>"""', '',
      'PARAMETER stop "<|im_end|>"',
      'PARAMETER stop "<|im_start|>"']
open('Modelfile','w').write('\n'.join(mf))
print('Wrote Modelfile. Downloading BOTH files — keep them in the SAME folder.')
files.download('Modelfile')
files.download(gguf)


### 8) Run it locally in Vio
Step 7 downloaded **two** files — the `.gguf` and a ready-made **`Modelfile`**. Put them
**in the same folder** on your PC. The Modelfile already has the correct Qwen chat template,
so you don't hand-write anything.

**a)** Open a terminal in that folder and register the model:
```
ollama create vio-net -f Modelfile
```
**b)** Point Vio at it and restart. In the same terminal you launch Vio from:
```
set VIO_LLM_MODEL=vio-net
python web.py
```
That's it — Vio now reasons through **your** fine-tuned model, fully local and offline.

---
**To improve it later:** add more datasets to `mind/datasets/`, then rerun this notebook from
step 4. More question→answer pairs = a sharper model. Fine-tune teaches *voice and shape*;
keep feeding facts through Vio's document library (retrieval) so they stay current.
